# Fourier Transform of the Trapezoidal Envelope $\alpha(t)$

This notebook derives $\hat{\alpha}(\omega)$ for the trapezoidal spot evolution $\alpha(t)$
by two routes: (i) direct integration and (ii) the convolution theorem.

**Key result:**
$$\hat{\alpha}(\omega) = \frac{2\alpha_{\max}}{\tau\omega^2}\left[\cos\!\left(\frac{\omega\ell}{2}\right) - \cos\!\left(\frac{\omega\ell}{2} + \omega\tau\right)\right]$$

The paper does not typeset $\hat{\alpha}(\omega)$ (its Appendix `app:fourier_trapezoid` transforms the squared envelope
$\Gamma = \alpha^2/\alpha_{\max}^2$; see `derive_fourier_gamma.ipynb`). It does typeset the trapezoid itself, in the main text
(Eq. `eqn:trapezoid`, Section `sec:spot_profile`) and in the appendix (Eq. `alpha_piecewise`); the $\alpha(t)$ used here
is checked against both printed forms, taken from the equation registry (`equations.py`), where each is transcribed
into SymPy once, exactly as typeset; `check_equations.py` keeps the transcriptions in sync with `ms.tex`.

In [ ]:
import sympy as sp
from sympy import pi, cos, sin, integrate, simplify, symbols, latex, oo, Rational
from IPython.display import display, Math
sp.init_printing()

In [ ]:
t = sp.Symbol('t', real=True)
w = sp.Symbol('omega', positive=True)
ell = sp.Symbol('ell', positive=True)       # plateau duration
tau = sp.Symbol('tau', positive=True)        # ramp duration
amax = sp.Symbol('alpha_max', positive=True)

## Direct integration

Since $\alpha(t)$ is real and even:
$$\hat{\alpha}(\omega) = 2\int_0^\infty \alpha(t)\cos(\omega t)\,dt = 2\alpha_{\max}\left[I_1 + I_2\right]$$
where $I_1$ is the plateau integral and $I_2$ is the falling ramp integral.

In [ ]:
# I_1: plateau integral (0 to ell/2)
I1 = integrate(cos(w * t), (t, 0, ell / 2))
I1 = simplify(I1)
print("I_1 (plateau):")
display(Math(rf"I_1 = {latex(I1)}"))

In [ ]:
# I_2: falling ramp (ell/2 to ell/2 + tau)
# alpha(t) = amax * (ell/2 + tau - t) / tau on this interval
ramp = (ell / 2 + tau - t) / tau
I2 = integrate(ramp * cos(w * t), (t, ell / 2, ell / 2 + tau))
I2 = simplify(I2)
print("I_2 (ramp):")
display(Math(rf"I_2 = {latex(I2)}"))

In [ ]:
# Combined: alpha_hat = 2 * amax * (I1 + I2)
alpha_hat_direct = 2 * amax * (I1 + I2)
alpha_hat_direct = simplify(alpha_hat_direct)
print("Direct result (Eq. alpha_hat_direct):")
display(Math(rf"\hat{{\alpha}}(\omega) = {latex(alpha_hat_direct)}"))

# Verify it matches the expected form
expected = 2 * amax / (tau * w**2) * (cos(w * ell / 2) - cos(w * ell / 2 + w * tau))
assert simplify(alpha_hat_direct - expected) == 0
print("\nVerified: matches expected form.")

### Check against the paper: the trapezoid of Eq. `alpha_piecewise`

The direct integration above uses the plateau and the falling ramp on $t \ge 0$ and the evenness of $\alpha(t)$. The
paper's five-branch $\alpha(t)$ (Appendix `app:fourier_trapezoid`, $\ell_{\rm spot} \to \ell$, $\tau_{\rm spot} \to \tau$) is taken from the
registry; its falling ramp must be the integrand used for $I_2$, it must be even and continuous, and its full complex
transform $\int\alpha(t)\,e^{-i\omega t}\,dt$, computed branch by branch without the evenness shortcut, must equal
$\hat{\alpha}(\omega)$ above.

In [ ]:
import equations as eqs   # the paper's equations, transcribed once (src/derivations/equations.py)

to_nb = {eqs.tau_s: tau}   # tau_spot is tau here; t, omega, ell and alpha_max are the registry's own symbols

alpha_paper = eqs.get("eq:alpha_piecewise").expr.subs(to_nb)   # Eq. alpha_piecewise
alpha_branches = [e for e, _ in (alpha_paper / amax).args]
assert len(alpha_branches) == 5
assert alpha_branches[0] == 0 and alpha_branches[2] == 1 and alpha_branches[4] == 0
alpha_rise_paper = amax * alpha_branches[1]    # -ell/2 - tau <= t < -ell/2
alpha_fall_paper = amax * alpha_branches[3]    #  ell/2 < t <= ell/2 + tau

assert simplify(alpha_fall_paper - amax * ramp) == 0, "ramp used for I_2 != Eq. alpha_piecewise"
assert simplify(alpha_rise_paper.subs(t, -t) - alpha_fall_paper) == 0           # even
assert simplify(alpha_rise_paper.subs(t, -ell / 2 - tau)) == 0 and simplify(alpha_rise_paper.subs(t, -ell / 2) - amax) == 0
assert simplify(alpha_fall_paper.subs(t, ell / 2) - amax) == 0 and simplify(alpha_fall_paper.subs(t, ell / 2 + tau)) == 0

phase = sp.exp(-sp.I * w * t)
alpha_hat_full = (integrate(alpha_rise_paper * phase, (t, -ell / 2 - tau, -ell / 2))
                  + integrate(amax * phase, (t, -ell / 2, ell / 2))
                  + integrate(alpha_fall_paper * phase, (t, ell / 2, ell / 2 + tau)))
assert simplify((alpha_hat_full - alpha_hat_direct).rewrite(sp.exp)) == 0, "complex FT of Eq. alpha_piecewise != alpha_hat"
print("Verified: Eq. alpha_piecewise is the (even, continuous) trapezoid integrated above, and its full transform is alpha_hat")

## Convolution-theorem route

The trapezoid is the convolution of two rectangular pulses:
$$\alpha(t) = \frac{\alpha_{\max}}{\tau}\,\mathrm{Rect}_{W_1}(t) * \mathrm{Rect}_{W_2}(t)$$
where $W_1 = \ell + \tau$ and $W_2 = \tau$.

By the convolution theorem: $\hat{\alpha}(\omega) = \frac{\alpha_{\max}}{\tau}\hat{\mathrm{Rect}}_{W_1}\cdot\hat{\mathrm{Rect}}_{W_2}$.

In [ ]:
# FT of rect function: Rect_W has FT = 2*sin(w*W/2)/w
W1 = ell + tau
W2 = tau

FT_rect1 = 2 * sin(w * W1 / 2) / w
FT_rect2 = 2 * sin(w * W2 / 2) / w

alpha_hat_conv = amax / tau * FT_rect1 * FT_rect2
print("Convolution route (sinc product form):")
display(Math(rf"\hat{{\alpha}}(\omega) = {latex(alpha_hat_conv)}"))

In [ ]:
# Apply product-to-sum: sin(A)*sin(B) = (1/2)[cos(A-B) - cos(A+B)]
alpha_hat_conv_expanded = sp.expand_trig(sp.expand(alpha_hat_conv))
alpha_hat_conv_expanded = simplify(alpha_hat_conv_expanded)
print("After product-to-sum:")
display(Math(rf"\hat{{\alpha}}(\omega) = {latex(alpha_hat_conv_expanded)}"))

# Verify: matches direct integration result
assert simplify(alpha_hat_conv_expanded - alpha_hat_direct) == 0
print("\nVerified: convolution route matches direct integration.")

## DC limit

$\hat{\alpha}(0) = \alpha_{\max}(\ell + \tau)$ (area under the trapezoid).

In [ ]:
# DC limit: take w -> 0 via Taylor expansion
dc_limit = sp.limit(alpha_hat_direct, w, 0)
print(f"DC limit: alpha_hat(0) = {dc_limit}")

# Verify: area = amax * (plateau + 2 * triangle) = amax * (ell + tau)
expected_area = amax * (ell + tau)
assert simplify(dc_limit - expected_area) == 0
print(f"Verified: equals area under trapezoid = alpha_max * (ell + tau)")

In [ ]:
# Numerical verification
import numpy as np

def trapezoid(t_arr, ell_n, tau_n, amax_n):
    abs_t = np.abs(t_arr)
    L = ell_n / 2
    S = L + tau_n
    result = np.zeros_like(abs_t)
    plateau = abs_t <= L
    ramp = (abs_t > L) & (abs_t <= S)
    result[plateau] = amax_n
    result[ramp] = amax_n * (S - abs_t[ramp]) / tau_n
    return result

def alpha_hat_numerical(w_arr, ell_n, tau_n, amax_n, nt=10000):
    S = ell_n / 2 + tau_n
    t_grid = np.linspace(-S, S, nt)
    alpha_t = trapezoid(t_grid, ell_n, tau_n, amax_n)
    result = np.array([np.trapezoid(alpha_t * np.cos(wv * t_grid), t_grid) for wv in w_arr])
    return result

# Test parameters
ell_n, tau_n, amax_n = 15.0, 3.0, 0.1
w_arr = np.linspace(0.01, 4.0, 500)

# Analytic
alpha_hat_analytic = 2 * amax_n / (tau_n * w_arr**2) * (
    np.cos(w_arr * ell_n / 2) - np.cos(w_arr * ell_n / 2 + w_arr * tau_n))

# Numerical
alpha_hat_num = alpha_hat_numerical(w_arr, ell_n, tau_n, amax_n)

max_err = np.max(np.abs(alpha_hat_analytic - alpha_hat_num))
print(f"Max error (analytic vs numerical): {max_err:.2e}")
assert max_err < 1e-4, "analytic alpha_hat disagrees with numerical integration"
print("Numerical verification passed.")

# The numpy trapezoid used for the quadrature is the paper's Eq. alpha_piecewise
alpha_paper_np = sp.lambdify((t, ell, tau, amax), alpha_paper, "numpy")
t_check = np.linspace(-(ell_n / 2 + tau_n) - 2, ell_n / 2 + tau_n + 2, 4001)
assert np.allclose(trapezoid(t_check, ell_n, tau_n, amax_n), alpha_paper_np(t_check, ell_n, tau_n, amax_n),
                   rtol=0, atol=1e-15), "numpy trapezoid != Eq. alpha_piecewise"

### Check against the paper: the main-text trapezoid, Eq. `eqn:trapezoid`

Section `sec:spot_profile` defines the trapezoid in the Heaviside form of Kipping (2012),
$$\frac{\alpha_{\rm trap}(t)}{\alpha_{\max}} = \mathcal{I}^{-1}[\Delta t_1\mathsf{H}(\Delta t_1) - \Delta t_2\mathsf{H}(\Delta t_2)]
- \mathcal{E}^{-1}[\Delta t_3\mathsf{H}(\Delta t_3) - \Delta t_4\mathsf{H}(\Delta t_4)],$$
with $\Delta t_1 = t - (t_{\rm ref} - \ell_{\rm spot}/2 - \tau_{\rm em})$, $\Delta t_2 = t - (t_{\rm ref} - \ell_{\rm spot}/2)$,
$\Delta t_3 = t - (t_{\rm ref} + \ell_{\rm spot}/2)$, $\Delta t_4 = t - (t_{\rm ref} + \ell_{\rm spot}/2 + \tau_{\rm dec})$,
and $\mathcal{I} = \tau_{\rm em}$, $\mathcal{E} = \tau_{\rm dec}$, which the text says are the maxima of the two bracketed ramps.
The registry's transcription has the $\Delta t_i$, $\mathcal{I}$ and $\mathcal{E}$ substituted. Checked:

1. it is the expression above with the $\Delta t_i$ and normalizations as the text defines them, and the bracketed
   ramps saturate at $\tau_{\rm em}$ and $\tau_{\rm dec}$, so $\mathcal{I}$ and $\mathcal{E}$ normalize the peak to 1;
2. for $\tau_{\rm em} = \tau_{\rm dec} = \tau$ and $t_{\rm ref} = 0$ it is exactly the $\alpha(t)/\alpha_{\max}$ of Eq. `alpha_piecewise`
   used above (symbolically, branch by branch);
3. for unequal timescales it is zero outside $[t_{\rm ref} - \ell_{\rm spot}/2 - \tau_{\rm em},\ t_{\rm ref} + \ell_{\rm spot}/2 + \tau_{\rm dec}]$,
   rises linearly over $\tau_{\rm em}$, holds 1 over the plateau $\ell_{\rm spot}$ centred on $t_{\rm ref}$, and decays linearly over $\tau_{\rm dec}$.

In [ ]:
from sympy import Heaviside

t_ref, tau_em, tau_dec = eqs.t_ref, eqs.tau_em, eqs.tau_dec
alpha_kip = eqs.get("eqn:trapezoid").expr      # Eq. eqn:trapezoid, alpha_trap / alpha_max

# 1. The Delta t_i and normalizations as the text defines them; the ramps saturate at I and E
#    (evaluated past each ramp), so the peak is 1
dt1 = t - (t_ref - ell / 2 - tau_em)
dt2 = t - (t_ref - ell / 2)
dt3 = t - (t_ref + ell / 2)
dt4 = t - (t_ref + ell / 2 + tau_dec)
rise_kip = dt1 * Heaviside(dt1) - dt2 * Heaviside(dt2)
fall_kip = dt3 * Heaviside(dt3) - dt4 * Heaviside(dt4)
I_norm, E_norm = tau_em, tau_dec            # as printed
assert simplify(alpha_kip - (rise_kip / I_norm - fall_kip / E_norm)) == 0
assert simplify(rise_kip.subs(t, t_ref) - I_norm) == 0
assert simplify(fall_kip.subs(t, t_ref + ell / 2 + tau_dec + 1) - E_norm) == 0
assert simplify(alpha_kip.subs(t, t_ref)) == 1

# 2. Symmetric case = Eq. alpha_piecewise, branch by branch: a point inside each branch fixes every
#    Heaviside factor, leaving a polynomial in t that must equal that branch of Eq. alpha_piecewise
alpha_sym = alpha_kip.subs({t_ref: 0, tau_em: tau, tau_dec: tau})
S = ell / 2 + tau
branches = [(-S - 1, 0), (-ell / 2 - tau / 2, alpha_rise_paper / amax), (0, 1),
            (ell / 2 + tau / 2, alpha_fall_paper / amax), (S + 1, 0)]
for t_in, paper_branch in branches:
    kip_branch = alpha_sym.replace(Heaviside, lambda x, *h0: Heaviside(x.subs(t, t_in)))
    assert not kip_branch.has(Heaviside)
    assert simplify(kip_branch - paper_branch) == 0, f"branch containing t = {t_in}"

# 3. Asymmetric shape on an exact grid (ell = 10, tau_em = 2, tau_dec = 5, t_ref = 3)
vals = {ell: 10, tau_em: 2, tau_dec: 5, t_ref: 3}
def expected(tt):
    if tt <= -4 or tt >= 13:
        return 0
    if tt < -2:
        return (tt + 4) / sp.Integer(2)
    if tt <= 8:
        return 1
    return (13 - tt) / sp.Integer(5)
grid = [sp.Rational(k, 4) for k in range(-32, 69)]
bad = [tt for tt in grid if simplify(alpha_kip.subs(vals).subs(t, tt) - expected(tt)) != 0]
assert not bad, f"Eq. eqn:trapezoid has the wrong asymmetric shape at t = {bad[:5]}"
print("Verified: Eq. eqn:trapezoid as printed is the plateaued trapezoid, and reduces to Eq. alpha_piecewise")